In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.4 Temperature and decoding

> 🎯 **Goal:** turn the model's probability distribution into actual text, and learn the knobs (temperature, top_k, top_p) that control how adventurous that text is.

**Why this matters.** In l0.3 you saw the model produce a distribution but never choose. *Decoding* is the choosing. The same model, with the same distribution, can be made to sound robotically safe or wildly creative depending only on how you sample. This is why the same chatbot can be boring one moment and surprising the next: the model did not change, the decoding settings did. Every product built on a language model tunes these knobs.

**The intuition.** Think of the distribution as a bag of raffle tickets, more tickets for likelier characters. *Temperature* changes how lopsided the bag is. Low temperature (near 0) means you almost always draw the single most popular ticket: safe, repetitive, predictable. High temperature flattens the odds so even rare tickets get drawn: surprising, sometimes nonsense. *top_k* and *top_p* take a different approach: instead of reshaping the odds, they throw away the long tail of unlikely tickets entirely, then draw from what remains.

**The mechanics.** Temperature `t` divides the model's raw scores before they become probabilities. Dividing by a small `t` exaggerates differences (the top choice dominates); dividing by a large `t` shrinks differences (choices even out). At `t=0` we skip sampling and just take the single highest-probability character every time; this is called *greedy* or *argmax* decoding, and because there is no randomness it is fully deterministic. *top_k* keeps only the k most likely characters and samples among them. *top_p* (also called nucleus sampling) keeps the smallest set of characters whose probabilities add up to p (say 0.9) and samples among those. The first cell sweeps temperature; the second contrasts top_k and top_p.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
for t in [0, 0.5, 1.0, 1.5]:
    print(f"--- temperature={t} ---")
    print(generate(model, tok, "The ", max_new_tokens=60, temperature=t, seed=0))

**What just happened (so far).** Reading the four blocks top to bottom, you watched the same model slide from rigid to unhinged. At `temperature=0` it is confident and repetitive (often looping on a few characters). At `0.5` it loosens slightly. At `1.0` it uses the model's beliefs as-is. At `1.5` it gets chaotic, reaching for rare characters and producing more noise. Same weights, same prompt, only the knob moved.

Now contrast a different family of knobs. `top_k` and `top_p` do not reshape the odds; they *clip the unlikely tail* before sampling, which keeps output coherent while still allowing variety:

In [ ]:
print("top_k=5: ", generate(model, tok, "The ", max_new_tokens=60, top_k=5, seed=0))
print("top_p=0.9:", generate(model, tok, "The ", max_new_tokens=60, top_p=0.9, seed=0))

In [ ]:
# Temperature 0 is greedy argmax, so it is fully deterministic.
a = generate(model, tok, "The ", max_new_tokens=30, temperature=0)
b = generate(model, tok, "The ", max_new_tokens=30, temperature=0)
assert a == b

**What just happened.** The two `temperature=0` runs came out identical, and the assert confirms it. Greedy decoding takes the single highest-probability character every time, so there is nothing random left to vary: same input, same output, forever. That determinism is useful (reproducible demos, comparable benchmarks) but it has a dark side you will exploit in the very next lesson: with nothing to break ties or escape a rut, greedy decoding can get stuck repeating itself.

⚠️ **Common pitfalls**
- Cranking temperature up for "more creativity" and getting garbage instead. Past about 1.2 the rare-character noise usually overwhelms any gain. Reach for top_k or top_p when you want variety without chaos.
- Thinking top_k and top_p are random in a bad way. They still sample, but only from the plausible characters, so they stay coherent while avoiding the repetition trap of greedy decoding.
- Expecting `temperature=0` runs to ever differ. They cannot. If you need variety, you need temperature above 0 (or top_k / top_p).

🏋️ **Try it yourself.** In the temperature-sweep cell, add `2.5` to the list `[0, 0.5, 1.0, 1.5]` and re-run to see decoding fall apart at very high heat. Then in the top_k / top_p cell, change `top_k=5` to `top_k=1` and predict what happens before running it (hint: top_k=1 is the same as greedy).